In [1]:
import warnings
warnings.filterwarnings('ignore')

(Tutorial_Get_halogen_bond_sites)=
# get_halogen_bond_sites

*Recognizing ordered chemical sites for directional halogen bonding.*

Prepare sites independently of interaction geometry. This experimental profile reproduces the chemical SMARTS of ProLIF 2.2.2. RDKit is optional and required for this calculation; ProLIF is not required.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

See {func}`molsysmt.physchem.get_halogen_bond_sites` for arguments, errors and returned data.
:::

## Basic usage

Let's show how this function works with a controlled halogen donor and carbonyl acceptor. Recognition requires declared elements, bonds, orders, formal charges and aromatic flags, but no coordinates.

In [2]:
import molsysmt as msm

In [3]:
from rdkit import Chem
molsys = msm.convert(Chem.MolFromSmiles('CCl.C=O'), to_form='molsysmt.MolSys')
sites = msm.physchem.get_halogen_bond_sites(molsys)
print('Donor–halogen:', sites['donor_halogen_pairs'])
print('Acceptor–reference:', sites['acceptor_reference_pairs'])

Donor–halogen: [[0 1]]
Acceptor–reference: [[3 2]]


## Preserving chemical roles

Each row has role order, not ascending atom order. Every eligible acceptor neighbor defines a separate pair. Fluorine is not a terminal donor in this profile; carbonyl double links are eligible, triple links and positively charged acceptors are excluded.

In [4]:
multi = msm.physchem.get_halogen_bond_sites(Chem.MolFromSmiles('CBr.COC'))
print(multi['acceptor_reference_pairs'])
print(msm.physchem.get_halogen_bond_sites(Chem.MolFromSmiles('CF.C[NH3+]'))['donor_halogen_pairs'].shape)

[[3 2]
 [3 4]]


(0, 2)


## Selecting complete sites

The full source graph is recognized first. Both atoms of a site must belong to the selection. Selecting only the acceptor does not retain its reference neighbor implicitly.

In [5]:
for selection in ([3], [2, 3]):
    selected = msm.physchem.get_halogen_bond_sites(molsys, selection=selection)
    print(selection, selected['acceptor_reference_pairs'])

[3] []


[2, 3] [[3 2]]


## Preparing ligand and peptide chemistry

A chlorinated aromatic ligand and an amide-containing fragment illustrate chemistry relevant to ligand recognition. This disconnected declared graph is a chemical control, not a binding pose or an energetic prediction.

In [6]:
fragments = Chem.MolFromSmiles('Clc1ccccc1.CC(=O)NCC(=O)O')
features = msm.physchem.get_halogen_bond_sites(fragments)
print(features['donor_halogen_pairs'])
print(features['acceptor_reference_pairs'])

[[1 0]]
[[ 1  0]
 [ 1  2]
 [ 1  6]
 [ 2  3]
 [ 3  4]
 [ 4  5]
 [ 5  6]
 [ 9  8]
 [10  8]
 [10 11]
 [13 12]
 [14 12]]


## Choosing chemical state and source form

The default reference state supplies chemistry. Select a stored state with `chemical_state=0`, or use `chemical_state="structure"` only when selected structures resolve one assigned state. Multiple states are not merged. RDKit, native chemistry and H5MSM are supported when they provide the required assignments; missing charge/aromatic data is not inferred silently.

In [7]:
native = msm.physchem.get_halogen_bond_sites(molsys, chemical_state=0, max_matches=100000)
external = msm.physchem.get_halogen_bond_sites(Chem.MolFromSmiles('CCl.C=O'))
assert (native['acceptor_reference_pairs'] == external['acceptor_reference_pairs']).all()
print(native['evidence'])

{'kind': 'declared_chemical_graph', 'connectivity_completeness': 'complete', 'assume_complete_connectivity': False, 'chemical_state_provenance_index': None}


## Recording limitations and provenance

`max_matches` is checked on full-source matching before selection. Exceeding it raises instead of truncating. `assume_complete_connectivity=True` records an explicit assumption; it does not repair connectivity or supply missing assignments. The bibliography credits the SMARTS reference, not a geometric calculation that was not performed.

In [8]:
print(sites['method_reference'])
print(sites['software'])
print([item['id'] for item in sites['attribution']['items']])

{'software': 'ProLIF', 'version': '2.2.2', 'commit': '19f1800218387c49536eb9d3e8cd3044fdb337ee', 'implementation': 'https://github.com/chemosim-lab/ProLIF/blob/19f1800218387c49536eb9d3e8cd3044fdb337ee/prolif/interactions/interactions.py', 'geometry': 'https://github.com/chemosim-lab/ProLIF/blob/19f1800218387c49536eb9d3e8cd3044fdb337ee/prolif/utils.py', 'paper': 'https://doi.org/10.1186/s13321-021-00548-6'}
{'molsysmt': '0.21.0+606.ga03eb4bf6', 'rdkit': '2025.09.5'}
['doi:10.1186/s13321-021-00548-6', 'software:molsysmt:0.21.0+606.ga03eb4bf6', 'software:rdkit:2025.09.5']


:::{seealso}
:class: dropdown

- {ref}`Tutorial_Convert` — converting molecular systems with {func}`molsysmt.basic.convert`.
- {ref}`Tutorial_Get_halogen_bonds` — calculating observations with {func}`molsysmt.interactions.halogen_bonds.get_halogen_bonds`.
:::